# Defense Evaluation at 140 Records

Re-runs the three privacy defenses at the 140-record scale so that Chapter 6 matches Chapter 5. The 140-record LoRA adapter ships inside this bundle, so the two-hour fine-tuning step is **skipped**.

**Before starting:** Runtime → Change runtime type → **L4 GPU** (T4 also works, roughly twice as slow).

Everything is written to Google Drive, and every stage checkpoints, so a disconnect costs only the stage in flight — just re-run the same cell.

## 1 · Confirm the GPU

In [ ]:
!nvidia-smi

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/piibench'
os.makedirs(DRIVE, exist_ok=True)
print('results will be written to:', DRIVE)

## 3 · Unpack the bundle
Upload `piibench_defense140.zip` to the **`piibench` folder** in your Drive first (drag it in from the Drive web page). This cell unpacks it into the session.

In [ ]:
import zipfile, os
src = f'{DRIVE}/piibench_defense140.zip'
assert os.path.exists(src), f'not found: {src} -- upload the zip to Drive first'
zipfile.ZipFile(src).extractall('/content/work')
os.chdir('/content/work')
print(sorted(os.listdir('.')))
print('adapter present:', os.path.exists('adapter/adapter_config.json'))

## 4 · Install dependencies
(~2 minutes.)

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets pandas

## 5 · Part 1 — output filtering and unlearning
Reuses the bundled adapter, so no fine-tuning. Skips the base arm, which was already measured at 0.000 for this corpus.

**About 1.5 hours on an L4.**

In [ ]:
!python defense_eval.py --model Qwen/Qwen2.5-1.5B --persons 140 \
  --adapter adapter --skip-dp --skip-base \
  --out-dir $DRIVE/defense_140

## 6 · Part 2 — differentially private fine-tuning
Same output directory, so Part 1 is detected and skipped; only DP-LoRA runs.

**About 3 hours on an L4** — DP-SGD computes one backward pass per example in order to clip per-example gradients, which is inherently slow.

If you would rather stop after Part 1, skip this cell and tell me; the DP figure from the 40-record run can be reported with a scope note instead.

In [ ]:
!python defense_eval.py --model Qwen/Qwen2.5-1.5B --persons 140 \
  --adapter adapter --skip-base \
  --out-dir $DRIVE/defense_140

## 7 · Offline analysis (seconds, no GPU)

In [ ]:
!python analyze_records.py $DRIVE/defense_140/records.csv \
  --out $DRIVE/defense_140/analysis

## 8 · Collect the results
The files are already in Drive. This cell just makes a small archive of the parts to send back (it excludes the multi-hundred-megabyte adapters).

In [ ]:
import shutil, os, glob
src = f'{DRIVE}/defense_140'
os.makedirs('/content/send', exist_ok=True)
for pat in ['results.json','summary.csv','per_category.csv','mer_long.csv',
            'records.csv','rec_*.csv']:
    for f in glob.glob(f'{src}/{pat}'):
        shutil.copy(f, '/content/send/')
if os.path.isdir(f'{src}/analysis'):
    shutil.copytree(f'{src}/analysis', '/content/send/analysis', dirs_exist_ok=True)
shutil.make_archive(f'{DRIVE}/defense140_results', 'zip', '/content/send')
print('wrote', f'{DRIVE}/defense140_results.zip',
      round(os.path.getsize(f'{DRIVE}/defense140_results.zip')/1e6, 1), 'MB')

---
### If the session drops
Re-run cells 2 → 3 → 4, then the cell you were on. Completed work is detected and skipped; you should see lines such as `resuming: N queries already on disk` or `already complete`.

### If you hit CUDA out of memory
Add `--batch-size 12` to the command.

### Watch for
`[C4]` should print `[unlearn] NNN LoRA tensors enabled for gradient ascent`, and `[C5]` prints a per-epoch progress line with an ETA.